# Deliverable 03a: Exploratory Data Analysis (EDA)

In this notebook, we perform extensive Exploratory Data Analysis to deeply understand our `composite-scam-transcript-dataset`. 

Our goal here is to **describe the data**, identify patterns that distinguish legitimate calls from scams, and inform our modeling decisions. Notably, because we are using a contextual Transformer model (DistilBERT), we intentionally avoid aggressive preprocessing (like stop-word removal or stemming) during the actual training phase, as these models rely heavily on natural language context. However, analyzing the raw data's structural and lexical properties is still crucial.

In [ ]:
!pip install wordcloud matplotlib seaborn scikit-learn pandas nltk -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic styling
sns.set_theme(style="whitegrid", palette="muted")


### Data Leakage & Provenance Audit

Below we check for any structural artifacts (such as leading quotes from merging datasets) that correlate highly with the scam label, and ensure no train/test overlap exists.

In [ ]:
import pandas as pd
import sys
sys.path.append("..")
from src.eda_utils import get_overlap_count, get_quote_artifact_stats

df_train_raw = pd.read_csv("../data/raw/composite_train.csv")
df_test_raw = pd.read_csv("../data/raw/composite_test.csv")

overlap = get_overlap_count(df_train_raw, df_test_raw, text_col="transcript")
print(f"Raw Train/Test Overlap (Leaks): {overlap} rows")

print("\nQuote Artifacts in Raw Data:")
stats = get_quote_artifact_stats(df_train_raw, text_col="transcript", label_col="is_scam")
display(stats)


## 1. Load the Raw Data

In [ ]:
# Assuming this is run locally or in Kaggle with the repo cloned
df = pd.read_csv('../data/raw/composite_train.csv')

# Ensure columns map correctly for our analysis
if "transcript" in df.columns and "is_scam" in df.columns:
    df = df.rename(columns={"transcript": "text", "is_scam": "label"})

print(f"Dataset Shape: {df.shape}")
display(df.head())


## 2. Basic Statistics and Class Imbalance
First, let's look at the distribution of our target variable (`label`). Are we dealing with a highly imbalanced dataset?

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='label', data=df)
plt.title('Distribution of Scam (1) vs Legitimate (0) Transcripts')
plt.xlabel('Label')
plt.ylabel('Count')

# Add counts above bars
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='bottom')
plt.show()

print("Percentage of Scam Transcripts: {:.2f}%".format((df['label'].mean()) * 100))


## 3. Sequence Length Distribution
Transformers have a maximum sequence length (usually 512 tokens). If our transcripts are significantly longer than this, they will be truncated, meaning the model might miss critical scam indicators at the end of a long call. Let's analyze the character and word counts.

In [ ]:
# Calculate word counts
df['word_count'] = df['text'].astype(str).apply(lambda x: len(x.split()))

plt.figure(figsize=(12, 5))

# Plot overall distribution
plt.subplot(1, 2, 1)
sns.histplot(df['word_count'], bins=50, kde=True, color='blue')
plt.title('Overall Word Count Distribution')
plt.xlabel('Number of Words')

# Plot by label
plt.subplot(1, 2, 2)
sns.boxplot(x='label', y='word_count', data=df)
plt.title('Word Count Distribution by Class')
plt.xlabel('Class (0=Legit, 1=Scam)')
plt.ylabel('Number of Words')

plt.tight_layout()
plt.show()

print(f"95th Percentile of word counts: {np.percentile(df['word_count'], 95):.0f}")


## 4. N-Gram Analysis (Bigrams)
What are the most common two-word phrases (bigrams) used in scams versus legitimate transcripts? This helps us understand the conversational structure.

In [ ]:
def get_top_ngrams(corpus, n=None, ngram_range=(2,2)):
    vec = CountVectorizer(stop_words='english', ngram_range=ngram_range).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:n]

scam_texts = df[df['label'] == 1]['text'].dropna()
legit_texts = df[df['label'] == 0]['text'].dropna()

top_scam_bigrams = get_top_ngrams(scam_texts, 15)
top_legit_bigrams = get_top_ngrams(legit_texts, 15)

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=[x[1] for x in top_legit_bigrams], y=[x[0] for x in top_legit_bigrams], ax=axes[0], palette="Greens_r")
axes[0].set_title('Top 15 Bigrams in Legitimate Calls')

sns.barplot(x=[x[1] for x in top_scam_bigrams], y=[x[0] for x in top_scam_bigrams], ax=axes[1], palette="Reds_r")
axes[1].set_title('Top 15 Bigrams in Scam Calls')

plt.tight_layout()
plt.show()


## 5. Word Cloud Visualizations
A qualitative look at the most frequent vocabulary.

In [ ]:
scam_text_combined = " ".join(scam_texts.astype(str))
legit_text_combined = " ".join(legit_texts.astype(str))

scam_wc = WordCloud(width=800, height=400, background_color='white', max_words=100, colormap='Reds').generate(scam_text_combined)
legit_wc = WordCloud(width=800, height=400, background_color='white', max_words=100, colormap='Greens').generate(legit_text_combined)

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

axes[0].imshow(legit_wc, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Legitimate Transcripts Word Cloud', fontsize=16)

axes[1].imshow(scam_wc, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Scam Transcripts Word Cloud', fontsize=16)

plt.tight_layout()
plt.show()


## 6. EDA Conclusions & Modeling Decisions
Based on the EDA above:
1. **Sequence Length**: The 95th percentile of word counts informs our decision to use a `max_length` of 256 or 512 in our DistilBERT Tokenizer. Most calls fit within this window, so truncation data loss is minimal.
2. **Vocabulary Overlap**: While Bigrams and Word Clouds show distinct patterns (e.g., urgency and financial keywords in scams), there is enough overlap in general vocabulary that simple TF-IDF or lexical matching is insufficient. We need contextual understanding.
3. **Preprocessing Strategy**: Because the linguistic structure (phrasing, urgency, tone) is critical to detecting manipulation, we will **NOT** strip stop words, lemmatize, or aggressively preprocess the text for the model. The `AutoTokenizer` will handle raw sequences directly to preserve the rich semantic context.